In [0]:
import os, sys, importlib

# ---- Parameters ----
dbutils.widgets.text("datasets", "installed_capacity,load_forecast,physical_flow", "Datasets (comma list)")
dbutils.widgets.text("max_rows", "400000", "Max rows per batch")

datasets = [s.strip() for s in dbutils.widgets.get("datasets").split(",") if s.strip()]
max_rows = int(dbutils.widgets.get("max_rows"))

# ---- Imports from the repo's src folder ----
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(os.path.join(repo_root, "src"))
import entsoe_parser, silver_loader
for m in (entsoe_parser, silver_loader):
    importlib.reload(m)

# ---- Load pending raw files into silver, one dataset at a time ----
for d in datasets:
    pending = silver_loader.pending_files(spark, d)
    print(f"{d}: {len(pending)} files pending (batches of up to {max_rows:,} rows)")
    print("   result:", silver_loader.run(spark, [d], max_rows=max_rows))

# ---- Silver row counts ----
display(spark.sql("""
    SELECT 'price' AS table_name, COUNT(*) AS silver_rows FROM power_prices.silver.price
    UNION ALL SELECT 'load',       COUNT(*) FROM power_prices.silver.load
    UNION ALL SELECT 'generation', COUNT(*) FROM power_prices.silver.generation
    UNION ALL SELECT 'flow',       COUNT(*) FROM power_prices.silver.flow
    UNION ALL SELECT 'capacity',   COUNT(*) FROM power_prices.silver.capacity
"""))